# 11 — Streamlit Execution Model & Session State

## 📓 Interactive Notebook · Module 07 · Intermediate

In this notebook, you'll learn:
1. **Why Streamlit reruns your script** on every interaction
2. **How `st.session_state` preserves data** across reruns
3. **Widget-session state integration** (bidirectional binding)
4. **Multi-step workflows** with proper state management
5. **Common state bugs** and how to fix them

---

## 📋 Objectives

By the end of this notebook, you will be able to:
- Explain Streamlit's top-to-bottom rerun execution model
- Initialize and persist values using `st.session_state`
- Build multi-step analysis workflows
- Debug common state-related bugs
- Implement user preference persistence

## 📋 Prerequisites

- Module 01–06 completed
- Basic Python knowledge (variables, loops, functions)
- Understanding of widgets and forms from Module 02

---

## 💡 The Execution Model — Why Reruns Happen

### The Core Concept

Streamlit executes your Python script **from top to bottom on every user interaction**. This is fundamentally different from traditional web frameworks.

Think of it like a **spreadsheet formula**: when any input changes, the formula recalculates. The formula itself doesn't "remember" previous calculations.

### 🎯 Visual: The Rerun Flow

```
User Interaction (click, type, select)
        │
        ▼
Widget value updates in browser
        │
        ▼
Streamlit sends new value to server
        │
        ▼
Script reruns from top to bottom
        │
        ▼
New page sent to browser
```

**Key insight:** Your entire script re-executes. Variables reset. Imports run again. Everything starts fresh — except `st.session_state`.

### The Problem Without Session State

In [ ]:
# ⚠️ THIS DOES NOT WORK AS EXPECTED
#
# Every time you click "Increment", the script reruns.
# The variable `counter` resets to 0 on each rerun.

import streamlit as st

# DON'T DO THIS:
# counter = 0  # Reset on every rerun!
# if st.button("Increment"):
#     counter += 1  # This only affects the local variable
# st.write(f"Counter: {counter}")  # Always shows "Counter: 1"

st.info("💡 Uncomment the code above to see the bug in action!")
st.write("The counter variable resets to 0 on every rerun, so it never actually counts.")

### The Solution: Session State

In [ ]:
import streamlit as st

# ✅ THIS WORKS - Use session state for persistence

# Step 1: Initialize (only runs once per session)
if "counter" not in st.session_state:
    st.session_state.counter = 0

# Step 2: Modify on interaction
col1, col2, col3 = st.columns(3)

with col1:
    if st.button("➕ Increment"):
        st.session_state.counter += 1

with col2:
    if st.button("➖ Decrement"):
        st.session_state.counter -= 1

with col3:
    if st.button("🔄 Reset"):
        st.session_state.counter = 0

# Step 3: Display current value
st.metric("Counter Value", st.session_state.counter)

st.success("✅ This counter persists across reruns!")

---

## 🔬 Experiment 1: Session State Basics

**Try it yourself:** Run this code and interact with the widgets. Then answer these questions:
1. What happens to `st.session_state.count` when you click different buttons?
2. How does the script "know" the current count without a local variable?

In [ ]:
import streamlit as st

# Initialize multiple session state values
if "count" not in st.session_state:
    st.session_state.count = 0
if "history" not in st.session_state:
    st.session_state.history = []

# Buttons
col1, col2 = st.columns(2)
with col1:
    if st.button("Add 1"):
        st.session_state.count += 1
        st.session_state.history.append(f"+1 → {st.session_state.count}")

with col2:
    if st.button("Subtract 1"):
        st.session_state.count -= 1
        st.session_state.history.append(f"-1 → {st.session_state.count}")

# Display
st.metric("Current Count", st.session_state.count)

if st.session_state.history:
    st.write("**History:**")
    for entry in st.session_state.history[-10:]:  # Show last 10
        st.write(f"  • {entry}")

---

## 💡 Widget-Session State Integration

### Bidirectional Binding

When a widget has a `key` parameter, it's automatically linked to `st.session_state[key]`:

```python
# These are equivalent:
name = st.text_input("Name", key="name")
name = st.session_state.name  # Same value!
```

### Setting Widget Values Programmatically

You can update widget values by setting their session state keys:

In [ ]:
import streamlit as st

# Initialize
if "user_name" not in st.session_state:
    st.session_state.user_name = ""
if "user_color" not in st.session_state:
    st.session_state.user_color = "Blue"

# Quick-fill buttons (set BEFORE the widget)
st.write("**Quick fill:**")
col1, col2 = st.columns(2)
with col1:
    if st.button("Fill as Alice"):
        st.session_state.user_name = "Alice"
with col2:
    if st.button("Fill as Bob"):
        st.session_state.user_name = "Bob"

# Widgets (read from session state)
name = st.text_input("Your name", key="user_name")
color = st.selectbox("Favorite color", 
                     ["Red", "Green", "Blue", "Purple"],
                     key="user_color")

# Display
st.write(f"Hello, **{name}**! Your favorite color is **{color}**.")

### ⚠️ The Order Rule

**Critical:** Set session state values **BEFORE** the widget renders, not after!

In [ ]:
import streamlit as st

# ✅ CORRECT: Set value BEFORE widget
if st.button("Reset Name"):
    st.session_state.name_correct = ""  # Set first

name_correct = st.text_input("Name (correct)", key="name_correct")
st.write(f"Current value: {name_correct}")

st.divider()

# ❌ INCORRECT: Set value AFTER widget
name_wrong = st.text_input("Name (wrong)", key="name_wrong")
if st.button("Reset Name (won't work this rerun)"):
    st.session_state.name_wrong = ""  # Too late! Shows old value until next rerun
st.write(f"Current value: {name_wrong}")

st.info("💡 The 'correct' version updates immediately; the 'wrong' version updates on the NEXT interaction.")

---

## 🔬 Experiment 2: Linked Widgets with Callbacks

**Try it yourself:** Change the category and observe how the subcategory updates automatically.

In [ ]:
import streamlit as st

# Data structure
categories = {
    "Fruits": ["Apple", "Banana", "Cherry", "Date"],
    "Vegetables": ["Carrot", "Broccoli", "Spinach"],
    "Grains": ["Rice", "Wheat", "Oats"]
}

# Initialize
if "category" not in st.session_state:
    st.session_state.category = "Fruits"
if "subcategory" not in st.session_state:
    st.session_state.subcategory = "Apple"

# Callback: when category changes, reset subcategory to first option
def on_category_change():
    st.session_state.subcategory = categories[st.session_state.category][0]

# Category selector (with callback)
category = st.selectbox(
    "Category",
    list(categories.keys()),
    key="category",
    on_change=on_category_change
)

# Subcategory selector (automatically reset)
subcategory = st.selectbox(
    "Subcategory",
    categories[st.session_state.category],
    key="subcategory"
)

st.write(f"Selected: **{category}** → **{subcategory}**")
st.write(f"\nSession state:")
st.json({"category": st.session_state.category, "subcategory": st.session_state.subcategory})

---

## 💡 Multi-Step Workflows

### Pattern: Step-by-Step Analysis Wizard

Build an app that guides users through:
1. Upload data
2. Select columns
3. Configure analysis
4. View results

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

# Initialize all session state
if "step" not in st.session_state:
    st.session_state.step = 1
if "df" not in st.session_state:
    st.session_state.df = None
if "selected_columns" not in st.session_state:
    st.session_state.selected_columns = []

# Progress bar
progress = st.progress(0, text=f"Step {st.session_state.step} of 4")

# --- Step 1: Upload ---
if st.session_state.step >= 1:
    st.header("📋 Step 1: Upload Data")
    uploaded = st.file_uploader("Upload a CSV file", type="csv", key="upload")
    
    if uploaded:
        st.session_state.df = pd.read_csv(uploaded)
        st.dataframe(st.session_state.df.head())
        st.write(f"Shape: {st.session_state.df.shape}")
        
        if st.button("Next →", key="next1"):
            st.session_state.step = 2
            st.rerun()
        progress.progress(25)

# --- Step 2: Select Columns ---
if st.session_state.step >= 2:
    st.header("📊 Step 2: Select Columns")
    st.session_state.selected_columns = st.multiselect(
        "Choose columns to analyze",
        st.session_state.df.columns.tolist(),
        default=st.session_state.df.columns[:3].tolist(),
        key="cols"
    )
    
    col1, col2 = st.columns(2)
    with col1:
        if st.button("← Back", key="back2"):
            st.session_state.step = 1
            st.rerun()
    with col2:
        if st.session_state.selected_columns:
            if st.button("Next →", key="next2"):
                st.session_state.step = 3
                st.rerun()
    progress.progress(50)

# --- Step 3: Analysis ---
if st.session_state.step >= 3:
    st.header("🔍 Step 3: Choose Analysis")
    analysis = st.selectbox(
        "What analysis?",
        ["Summary Statistics", "Correlation Matrix", "Distribution"],
        key="analysis"
    )
    
    col1, col2 = st.columns(2)
    with col1:
        if st.button("← Back", key="back3"):
            st.session_state.step = 2
            st.rerun()
    with col2:
        if st.button("Next →", key="next3"):
            st.session_state.step = 4
            st.rerun()
    progress.progress(75)

# --- Step 4: Results ---
if st.session_state.step >= 4:
    st.header("📈 Step 4: Results")
    subset = st.session_state.df[st.session_state.selected_columns]
    
    if st.session_state.get("analysis") == "Summary Statistics":
        st.dataframe(subset.describe())
    elif st.session_state.get("analysis") == "Correlation Matrix":
        st.dataframe(subset.select_dtypes(include=np.number).corr())
    else:  # Distribution
        for col in subset.select_dtypes(include=np.number).columns:
            st.bar_chart(subset[col].value_counts())
    
    col1, col2 = st.columns(2)
    with col1:
        if st.button("← Back", key="back4"):
            st.session_state.step = 3
            st.rerun()
    with col2:
        if st.button("🔄 Start Over"):
            st.session_state.step = 1
            st.session_state.df = None
            st.session_state.selected_columns = []
            st.rerun()
    progress.progress(100)

st.caption(f"Current step: {st.session_state.step}/4")

---

## 🔬 Experiment 3: User Preferences

**Try it yourself:** Change the preferences and observe how they persist across interactions.

In [ ]:
import streamlit as st

# Initialize preferences with defaults
default_prefs = {
    "theme": "light",
    "items_per_page": 10,
    "show_welcome": True,
    "language": "English"
}

if "prefs" not in st.session_state:
    st.session_state.prefs = default_prefs.copy()

# Sidebar for settings
with st.sidebar:
    st.header("⚙️ Preferences")
    
    st.session_state.prefs["theme"] = st.selectbox(
        "Theme", ["light", "dark", "blue"],
        index=["light", "dark", "blue"].index(st.session_state.prefs["theme"])
    )
    
    st.session_state.prefs["items_per_page"] = st.slider(
        "Items per page", 5, 50, st.session_state.prefs["items_per_page"]
    )
    
    st.session_state.prefs["show_welcome"] = st.toggle(
        "Show welcome message", st.session_state.prefs["show_welcome"]
    )
    
    if st.button("Reset to Defaults"):
        st.session_state.prefs = default_prefs.copy()
        st.rerun()

# Main content uses preferences
if st.session_state.prefs["show_welcome"]:
    st.balloons()
    st.success("👋 Welcome! Check the sidebar to customize your experience.")

st.header(f"Dashboard ({st.session_state.prefs['theme']} theme)")

# Generate sample data based on items_per_page
items = list(range(1, st.session_state.prefs["items_per_page"] + 1))
st.write(f"Showing {len(items)} items")
st.write(items)

st.json(st.session_state.prefs)

---

## 💡 Prediction History Pattern

Build an app that accumulates predictions across interactions:

In [ ]:
import streamlit as st
import pandas as pd
from datetime import datetime

# Initialize
if "predictions" not in st.session_state:
    st.session_state.predictions = []

# Input form
st.header("🔢 Simple Calculator")
col1, col2, col3 = st.columns(3)

with col1:
    num1 = st.number_input("First number", value=0.0, key="n1")
with col2:
    operation = st.selectbox("Operation", ["+", "-", "*", "/"], key="op")
with col3:
    num2 = st.number_input("Second number", value=1.0, key="n2")

if st.button("Calculate"):
    try:
        if operation == "+":
            result = num1 + num2
        elif operation == "-":
            result = num1 - num2
        elif operation == "*":
            result = num1 * num2
        else:  # division
            if num2 == 0:
                st.error("Cannot divide by zero!")
                st.stop()
            result = num1 / num2
        
        # Save to history
        st.session_state.predictions.append({
            "expression": f"{num1} {operation} {num2}",
            "result": result,
            "timestamp": datetime.now().strftime("%H:%M:%S")
        })
        st.success(f"**Result:** {result}")
    except Exception as e:
        st.error(f"Error: {e}")

# Display history
if st.session_state.predictions:
    st.divider()
    st.subheader("📜 Calculation History")
    
    df = pd.DataFrame(st.session_state.predictions)
    st.dataframe(df, use_container_width=True)
    
    if st.button("Clear History"):
        st.session_state.predictions = []
        st.rerun()

---

## ⚠️ Common Mistakes

### Mistake 1: Using Local Variables for State

```python
# ❌ WRONG
count = 0
if st.button("Add"):
    count += 1
st.write(count)  # Always shows 1

# ✅ CORRECT
if "count" not in st.session_state:
    st.session_state.count = 0
if st.button("Add"):
    st.session_state.count += 1
st.write(st.session_state.count)
```

### Mistake 2: Setting Value After Widget

```python
# ❌ WRONG
st.text_input("Name", key="name")
st.session_state.name = ""  # Too late!

# ✅ CORRECT
st.session_state.name = ""  # Set first
st.text_input("Name", key="name")
```

### Mistake 3: Widget in Conditional Block

```python
# ❌ WRONG - Widget may not render
if st.checkbox("Show advanced"):
    st.slider("Threshold", key="thresh")

# ✅ CORRECT - Always render, disable when needed
show = st.checkbox("Show advanced")
st.slider("Threshold", key="thresh", disabled=not show)
```

In [ ]:
# Example: Debug the bug below
import streamlit as st

st.header("🐛 Debug Challenge")

# BUG: This counter doesn't work correctly!
# Fix it using session_state

st.write("Fix the counter below so it actually counts:")

# TODO: Fix this code
# counter = 0
# if st.button("Increment"):
#     counter += 1
# st.write(f"Count: {counter}")

---

## 🎯 Challenges

### Challenge 1: Shopping Cart
Build a simple shopping cart that:
- Lets users add items from a dropdown
- Shows the cart contents
- Calculates the total
- Allows removing items

### Challenge 2: Note Taker
Build a note-taking app that:
- Lets users type notes
- Stores multiple notes in session state
- Shows a list of all notes
- Allows deleting individual notes

### Challenge 3: Color Palette Generator
Build a color palette app that:
- Lets users pick a base color
- Generates complementary colors
- Saves palettes to history
- Shows all saved palettes

In [ ]:
# Challenge 1: Shopping Cart
# TODO: Implement a shopping cart with session state

import streamlit as st

# Your code here


---

## 📝 Key Takeaways

1. **Streamlit reruns top-to-bottom on every interaction** — this is the core execution model.

2. **`st.session_state` preserves data across reruns** — without it, all local variables reset.

3. **Widget keys create bidirectional binding** with session state.

4. **Callbacks run before the rest of the script** — useful for linked widget updates.

5. **Multi-step workflows** use session state to track progress and accumulate results.

6. **Order matters** — set session state values **before** widgets that depend on them.

7. **Initialize early** — check and set all required keys at the top of your script.

### Common Mistakes Cheat Sheet

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Local variable for state | Value resets on rerun | Use `st.session_state` |
| Setting after widget | Widget doesn't update | Set before widget renders |
| Same key on two widgets | Conflicting values | Use unique keys |
| Widget in conditional | Value lost when hidden | Use `disabled` parameter |

---

## 📚 Further Reading

- [Streamlit Session State Docs](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state)
- [Execution Model](https://docs.streamlit.io/develop/concepts/architecture/exec-model)
- [Caching](https://docs.streamlit.io/develop/concepts/architecture/caching)

---

## 🔗 Related Materials

- 📖 Reading: [11 — Session State and Execution](../readings/11_session_state_and_execution.md)
- ✏️ Exercise: [11 — State Management Workshop](../exercises/11_state_management_workshop.py)
- 🖥️ Demo App: [11 — Session State Demo](../apps/11_session_state_demo.py)
- 📝 Quiz: [07 — Session State](../quizzes/07_session_state.md)
- 📖 Reading: [04 — Widget Keys and Behavior](../readings/04_widget_keys_and_behavior.md)